In [11]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Load processed data
sequences = np.load("../data/processed/sequences_normalized.npy")
labels = np.load("../data/processed/labels.npy", allow_pickle=True)

print(f"Sequences shape: {sequences.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Unique subjects: {len(np.unique(labels))}")
print(f"Data type: {sequences.dtype}")
print(f"Sample labels: {labels[:5]}")

Sequences shape: (20400, 11, 3)
Labels shape: (20400,)
Unique subjects: 51
Data type: float32
Sample labels: ['s002' 's002' 's002' 's002' 's002']


In [12]:
from sklearn.preprocessing import LabelEncoder

# Convert string labels to integers
le = LabelEncoder()
labels_encoded = le.fit_transform(labels)

print(f"Original labels (first 5): {labels[:5]}")
print(f"Encoded labels (first 5): {labels_encoded[:5]}")
print(f"Label range: {labels_encoded.min()} to {labels_encoded.max()}")
print(f"Number of classes: {len(le.classes_)}")
print(f"\nLabel mapping (first 5):")
for i, subject in enumerate(le.classes_[:5]):
    print(f"  {subject} -> {i}")

Original labels (first 5): ['s002' 's002' 's002' 's002' 's002']
Encoded labels (first 5): [0 0 0 0 0]
Label range: 0 to 50
Number of classes: 51

Label mapping (first 5):
  s002 -> 0
  s003 -> 1
  s004 -> 2
  s005 -> 3
  s007 -> 4


In [13]:
import torch
import torch.nn as nn

class CNNLSTM(nn.Module):
    def __init__(self, input_size=3, num_classes=51):
        super(CNNLSTM, self).__init__()
        
        # CNN layer: extracts local patterns across timesteps
        self.conv1 = nn.Conv1d(
            in_channels=input_size,
            out_channels=64,
            kernel_size=3,
            padding=1
        )
        self.relu = nn.ReLU()
        self.dropout1 = nn.Dropout(p=0.3)
        
        # LSTM layer: captures temporal dependencies across the sequence
        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )
        
        # Fully connected output layer
        self.fc = nn.Linear(128, num_classes)
    
    def forward(self, x):
        # x shape coming in: (batch, timesteps, features) = (batch, 11, 3)
        
        # Conv1d expects (batch, features, timesteps) so we permute
        x = x.permute(0, 2, 1)          # -> (batch, 3, 11)
        x = self.relu(self.conv1(x))     # -> (batch, 64, 11)
        x = self.dropout1(x)
        
        # LSTM expects (batch, timesteps, features) so we permute back
        x = x.permute(0, 2, 1)          # -> (batch, 11, 64)
        lstm_out, _ = self.lstm(x)       # -> (batch, 11, 128)
        
        # Take only the last timestep output
        x = lstm_out[:, -1, :]          # -> (batch, 128)
        
        # Final classification
        x = self.fc(x)                  # -> (batch, 51)
        return x

# Instantiate and inspect the model
model = CNNLSTM(input_size=3, num_classes=51)
print(model)
print(f"\nTotal trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

CNNLSTM(
  (conv1): Conv1d(3, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (relu): ReLU()
  (dropout1): Dropout(p=0.3, inplace=False)
  (lstm): LSTM(64, 128, num_layers=2, batch_first=True, dropout=0.3)
  (fc): Linear(in_features=128, out_features=51, bias=True)
)

Total trainable parameters: 238,643


In [14]:
# Create a fake batch of 32 samples to test the forward pass
dummy_input = torch.randn(32, 11, 3)  # (batch=32, timesteps=11, features=3)

model.eval()
with torch.no_grad():
    output = model(dummy_input)

print(f"Input shape:  {dummy_input.shape}")
print(f"Output shape: {output.shape}")
print(f"Expected:     (32, 51)")
print(f"Match: {output.shape == torch.Size([32, 51])}")

Input shape:  torch.Size([32, 11, 3])
Output shape: torch.Size([32, 51])
Expected:     (32, 51)
Match: True


In [15]:
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# ── Dataset class ──────────────────────────────────────────────────────────
class KeystrokeDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = torch.tensor(sequences, dtype=torch.float32)
        self.labels    = torch.tensor(labels, dtype=torch.long)
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

# ── Train / validation / test split ───────────────────────────────────────
# Split: 70% train, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    sequences, labels_encoded,
    test_size=0.30,
    random_state=42,
    stratify=labels_encoded
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print(f"Train size:      {X_train.shape[0]} samples")
print(f"Validation size: {X_val.shape[0]} samples")
print(f"Test size:       {X_test.shape[0]} samples")
print(f"Total:           {X_train.shape[0] + X_val.shape[0] + X_test.shape[0]} samples")

# Verify stratification held
import numpy as np
print(f"\nUnique subjects in train: {len(np.unique(y_train))}")
print(f"Unique subjects in val:   {len(np.unique(y_val))}")
print(f"Unique subjects in test:  {len(np.unique(y_test))}")

# ── DataLoaders ────────────────────────────────────────────────────────────
train_dataset = KeystrokeDataset(X_train, y_train)
val_dataset   = KeystrokeDataset(X_val,   y_val)
test_dataset  = KeystrokeDataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

print(f"\nTrain batches:      {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches:       {len(test_loader)}")

Train size:      14280 samples
Validation size: 3060 samples
Test size:       3060 samples
Total:           20400 samples

Unique subjects in train: 51
Unique subjects in val:   51
Unique subjects in test:  51

Train batches:      224
Validation batches: 48
Test batches:       48


In [ ]:
import torch.optim as optim
import matplotlib.pyplot as plt

# ── Training setup ─────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

model = CNNLSTM(input_size=3, num_classes=51).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

# ── Training loop ──────────────────────────────────────────────────────────
NUM_EPOCHS = 50
best_val_loss = float('inf')
best_model_state = None

train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

for epoch in range(NUM_EPOCHS):
    
    # ── Training phase ──
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for sequences_batch, labels_batch in train_loader:
        sequences_batch = sequences_batch.to(device)
        labels_batch    = labels_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(sequences_batch)
        loss    = criterion(outputs, labels_batch)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted  = torch.max(outputs, 1)
        total        += labels_batch.size(0)
        correct      += (predicted == labels_batch).sum().item()
    
    train_loss = running_loss / len(train_loader)
    train_acc  = correct / total
    
    # ── Validation phase ──
    model.eval()
    val_loss_sum, val_correct, val_total = 0.0, 0, 0
    
    with torch.no_grad():
        for sequences_batch, labels_batch in val_loader:
            sequences_batch = sequences_batch.to(device)
            labels_batch    = labels_batch.to(device)
            
            outputs      = model(sequences_batch)
            loss         = criterion(outputs, labels_batch)
            val_loss_sum += loss.item()
            
            _, predicted  = torch.max(outputs, 1)
            val_total    += labels_batch.size(0)
            val_correct  += (predicted == labels_batch).sum().item()
    
    val_loss = val_loss_sum / len(val_loader)
    val_acc  = val_correct / val_total
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        best_model_state = model.state_dict().copy()
    
    # Step scheduler
    scheduler.step(val_loss)
    
    # Record history
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    # Print every 5 epochs
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1:2d}/{NUM_EPOCHS}] "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

print(f"\nBest validation loss: {best_val_loss:.4f}")

# Save best model
import os
os.makedirs("../models", exist_ok=True)
torch.save(best_model_state, "../models/cnn_lstm_best.pt")
print("Best model saved to ../models/cnn_lstm_best.pt")

Training on: cpu
Epoch [ 5/50] Train Loss: 0.9175 | Train Acc: 0.7306 | Val Loss: 0.7535 | Val Acc: 0.7843
Epoch [10/50] Train Loss: 0.4925 | Train Acc: 0.8554 | Val Loss: 0.4444 | Val Acc: 0.8745
Epoch [15/50] Train Loss: 0.3185 | Train Acc: 0.9013 | Val Loss: 0.3560 | Val Acc: 0.8944
Epoch [20/50] Train Loss: 0.2239 | Train Acc: 0.9290 | Val Loss: 0.2972 | Val Acc: 0.9160
Epoch [25/50] Train Loss: 0.1527 | Train Acc: 0.9504 | Val Loss: 0.2932 | Val Acc: 0.9180
Epoch [30/50] Train Loss: 0.1150 | Train Acc: 0.9620 | Val Loss: 0.2733 | Val Acc: 0.9229
Epoch [35/50] Train Loss: 0.0705 | Train Acc: 0.9799 | Val Loss: 0.2595 | Val Acc: 0.9275
Epoch [40/50] Train Loss: 0.0515 | Train Acc: 0.9863 | Val Loss: 0.2555 | Val Acc: 0.9268


In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax1.plot(range(1, NUM_EPOCHS+1), train_losses, label='Train Loss', color='blue')
ax1.plot(range(1, NUM_EPOCHS+1), val_losses,   label='Val Loss',   color='orange')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('CNN-LSTM: Training vs Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy curves
ax2.plot(range(1, NUM_EPOCHS+1), train_accs, label='Train Accuracy', color='blue')
ax2.plot(range(1, NUM_EPOCHS+1), val_accs,   label='Val Accuracy',   color='orange')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('CNN-LSTM: Training vs Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../models/cnn_lstm_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Training curves saved.")

In [ ]:
import subprocess
result = subprocess.run(
    ['git', 'add', '.'],
    cwd='..',
    capture_output=True,
    text=True
)
print(result.stdout)
print(result.stderr)

result2 = subprocess.run(
    ['git', 'commit', '-m', 
     'Add CNN-LSTM baseline: model architecture, training loop, curves saved'],
    cwd='..',
    capture_output=True,
    text=True
)
print(result2.stdout)
print(result2.stderr)

result3 = subprocess.run(
    ['git', 'push', 'origin', 'main'],
    cwd='..',
    capture_output=True,
    text=True
)
print(result3.stdout)
print(result3.stderr)

In [ ]:
from sklearn.metrics import roc_curve
import numpy as np

# Load best model
model.load_state_dict(torch.load("../models/cnn_lstm_best.pt", 
                                  map_location=device))
model.eval()

# ── Collect scores and verification labels ─────────────────────────────────
all_genuine_scores  = []
all_impostor_scores = []

with torch.no_grad():
    for sequences_batch, labels_batch in test_loader:
        sequences_batch = sequences_batch.to(device)
        labels_batch    = labels_batch.to(device)
        
        outputs = model(sequences_batch)
        probs   = torch.softmax(outputs, dim=1)
        
        for i in range(len(labels_batch)):
            true_label    = labels_batch[i].item()
            genuine_score = probs[i, true_label].item()
            all_genuine_scores.append(genuine_score)
            
            # Impostor: pick a random different subject's score
            impostor_label = (true_label + 1) % 51
            impostor_score = probs[i, impostor_label].item()
            all_impostor_scores.append(impostor_score)

# ── Compute FAR, FRR, EER ──────────────────────────────────────────────────
genuine_scores  = np.array(all_genuine_scores)
impostor_scores = np.array(all_impostor_scores)

# Combine into binary verification problem
scores = np.concatenate([genuine_scores, impostor_scores])
labels_verification = np.concatenate([
    np.ones(len(genuine_scores)),   # 1 = genuine
    np.zeros(len(impostor_scores))  # 0 = impostor
])

# Compute ROC curve
fpr, tpr, thresholds = roc_curve(labels_verification, scores)
fnr = 1 - tpr  # FRR = 1 - TPR

# Find EER: point where FAR ≈ FRR
eer_idx = np.argmin(np.abs(fpr - fnr))
eer      = (fpr[eer_idx] + fnr[eer_idx]) / 2
eer_threshold = thresholds[eer_idx]

print(f"CNN-LSTM Baseline Results on Test Set")
print(f"{'='*45}")
print(f"Equal Error Rate (EER):  {eer*100:.2f}%")
print(f"EER Threshold:           {eer_threshold:.4f}")
print(f"FAR at EER:              {fpr[eer_idx]*100:.2f}%")
print(f"FRR at EER:              {fnr[eer_idx]*100:.2f}%")
print(f"\nGenuine score mean:      {genuine_scores.mean():.4f}")
print(f"Impostor score mean:     {impostor_scores.mean():.4f}")
print(f"Score separation:        {genuine_scores.mean() - impostor_scores.mean():.4f}")

In [ ]:
from sklearn.metrics import roc_curve
import numpy as np

# Load best model
model.load_state_dict(torch.load("../models/cnn_lstm_best.pt",
                                  map_location=device))
model.eval()

# ── Collect scores with multiple random impostors ──────────────────────────
np.random.seed(42)
NUM_IMPOSTOR_ATTEMPTS = 5  # test each genuine sample against 5 random impostors

all_genuine_scores  = []
all_impostor_scores = []

with torch.no_grad():
    for sequences_batch, labels_batch in test_loader:
        sequences_batch = sequences_batch.to(device)
        labels_batch    = labels_batch.to(device)

        outputs = model(sequences_batch)
        probs   = torch.softmax(outputs, dim=1)

        for i in range(len(labels_batch)):
            true_label    = labels_batch[i].item()
            genuine_score = probs[i, true_label].item()
            all_genuine_scores.append(genuine_score)

            # Sample multiple random impostors (excluding true label)
            possible_impostors = [j for j in range(51) if j != true_label]
            sampled_impostors  = np.random.choice(
                possible_impostors, 
                size=NUM_IMPOSTOR_ATTEMPTS, 
                replace=False
            )
            for imp_label in sampled_impostors:
                impostor_score = probs[i, imp_label].item()
                all_impostor_scores.append(impostor_score)

# ── Compute FAR, FRR, EER ──────────────────────────────────────────────────
genuine_scores  = np.array(all_genuine_scores)
impostor_scores = np.array(all_impostor_scores)

# Replicate genuine scores to match impostor count for fair comparison
genuine_scores_expanded = np.repeat(genuine_scores, NUM_IMPOSTOR_ATTEMPTS)

scores = np.concatenate([genuine_scores_expanded, impostor_scores])
labels_verification = np.concatenate([
    np.ones(len(genuine_scores_expanded)),
    np.zeros(len(impostor_scores))
])

# Compute ROC curve
fpr, tpr, thresholds = roc_curve(labels_verification, scores)
fnr = 1 - tpr

# Find EER
eer_idx       = np.argmin(np.abs(fpr - fnr))
eer           = (fpr[eer_idx] + fnr[eer_idx]) / 2
eer_threshold = thresholds[eer_idx]

print(f"CNN-LSTM Baseline Results (Rigorous Evaluation)")
print(f"{'='*50}")
print(f"Impostor attempts per genuine sample: {NUM_IMPOSTOR_ATTEMPTS}")
print(f"Total genuine scores:                 {len(genuine_scores_expanded):,}")
print(f"Total impostor scores:                {len(impostor_scores):,}")
print(f"\nEqual Error Rate (EER):  {eer*100:.2f}%")
print(f"EER Threshold:           {eer_threshold:.4f}")
print(f"FAR at EER:              {fpr[eer_idx]*100:.2f}%")
print(f"FRR at EER:              {fnr[eer_idx]*100:.2f}%")
print(f"\nGenuine score mean:      {genuine_scores.mean():.4f}")
print(f"Impostor score mean:     {impostor_scores.mean():.4f}")
print(f"Score separation:        {genuine_scores.mean() - impostor_scores.mean():.4f}")